# UAAP Season 87 Women's Volleyball: Exploratory Data Analysis
This notebook reproduces the data checks and the six EDA figures used in the paper. It runs in Google Colab or on a normal computer (needs pandas, numpy, matplotlib, and scikit-learn 1.6.1). Place `UAAP_Season87_Womens_Volleyball_Full__1_.csv` next to the notebook or in the repo root.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_NAME = "UAAP_Season87_Womens_Volleyball_Full__1_.csv"

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

candidates = [CSV_NAME, os.path.join("..", CSV_NAME), os.path.join("data", CSV_NAME)]
found = [p for p in candidates if os.path.exists(p)]
if not found and IN_COLAB:
    uploaded = files.upload()
    found = [n for n in uploaded if n.lower().endswith(".csv")]
if not found:
    raise FileNotFoundError(f"Could not find {CSV_NAME}.")
CSV_PATH = found[0]

os.makedirs("figures", exist_ok=True)
df = pd.read_csv(CSV_PATH)
numeric = ["Team_Wins_Before", "Opp_Wins_Before", "Team_Win_Streak", "Opp_Win_Streak"]
print("Using:", CSV_PATH)

## 1. Data quality checks

In [ ]:
print("Shape (rows, columns):", df.shape)
print(df.dtypes.to_string())
print("\nMissing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Team names consistent across Team and Opponent:", set(df.Team) == set(df.Opponent), "| teams:", df.Team.nunique())
print("Rows where Team equals Opponent:", int((df.Team == df.Opponent).sum()))
print("\nSummary statistics:")
print(df[numeric].describe().loc[["mean", "std", "min", "50%", "max"]].round(2).T)

## 2. Figure 1: Target class distribution
Every match appears twice (one winning and one losing row), so the classes are exactly balanced.

In [ ]:
counts = df.Match_Outcome.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Loss (0)", "Win (1)"], counts.values, color=["#3D6A85", "#45A878"])
ax.bar_label(bars)
ax.set_ylabel("Count"); ax.set_xlabel("Match Outcome (0 = Loss, 1 = Win)")
ax.set_title("Target Class Distribution (Match Outcome)")
plt.tight_layout(); plt.savefig("figures/fig1_class_distribution.png", dpi=200); plt.show()

## 3. Figure 2: Win streak vs win rate
Some streak values have very few records, so read the bars with caution.

In [ ]:
by_streak = df.groupby("Team_Win_Streak").Match_Outcome.agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_streak.index, by_streak["mean"], color="#4895C8")
ax.set_xticks(range(-13, 6)); ax.set_ylim(0, 1)
ax.set_xlabel("Team Win Streak"); ax.set_ylabel("Win Rate"); ax.set_title("Win Streak vs Win Rate")
plt.tight_layout(); plt.savefig("figures/fig2_streak_vs_win_rate.png", dpi=200); plt.show()
print(by_streak.T.round(3).to_string())
winning = df[df.Team_Win_Streak > 0]; losing = df[df.Team_Win_Streak < 0]
print(f"\nOn a winning streak: {winning.Match_Outcome.mean():.0%} won ({winning.Match_Outcome.sum()} of {len(winning)})")
print(f"On a losing streak:  {losing.Match_Outcome.mean():.0%} won ({losing.Match_Outcome.sum()} of {len(losing)})")

## 4. Figure 3: Pre-match wins for winning vs losing teams

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
data = [df[df.Match_Outcome == 0].Team_Wins_Before, df[df.Match_Outcome == 1].Team_Wins_Before]
bp = ax.boxplot(data, tick_labels=["Loss", "Win"], patch_artist=True, medianprops=dict(color="#333333"))
for patch, c in zip(bp["boxes"], ["#D4604F", "#43B777"]): patch.set_facecolor(c)
ax.set_xlabel("Match Outcome"); ax.set_ylabel("Team Wins Before Match")
ax.set_title("Pre-Match Win Count: Winners vs Losers")
plt.tight_layout(); plt.savefig("figures/fig3_wins_boxplot.png", dpi=200); plt.show()
print(df.groupby("Match_Outcome").Team_Wins_Before.median().to_dict(), "(median prior wins by outcome)")

## 5. Figure 4: Correlation heatmap

In [ ]:
corr = df[numeric + ["Match_Outcome"]].corr()
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("Correlation Heatmap of Numeric Features")
plt.tight_layout(); plt.savefig("figures/fig4_correlation_heatmap.png", dpi=200); plt.show()
print(corr.Match_Outcome.round(2).to_dict())

## 6. Figure 5: Random Forest feature importance
The model is fitted on the first 96 rows (48 matches) only, the same training set and pipeline used in the training notebook. Importances of the one-hot columns are summed back into their original features.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

features = ["Team", "Opponent", "Round"] + numeric
X, y = df[features], df["Match_Outcome"]
split = int(len(df) * 0.8)
X_train, y_train = X.iloc[:split], y.iloc[:split]

pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), ["Team", "Opponent", "Round"]),
])
rf = Pipeline([("preprocessor", pre), ("classifier", RandomForestClassifier(random_state=42))]).fit(X_train, y_train)

names = rf.named_steps["preprocessor"].get_feature_names_out()
imp = rf.named_steps["classifier"].feature_importances_
grouped = {}
for n, v in zip(names, imp):
    kind, n = n.split("__", 1)
    if kind == "cat":  # one-hot columns look like Team_Adamson, Opponent_UE, Round_Finals
        key = next(f for f in ["Opponent", "Team", "Round"] if n.startswith(f + "_"))
    else:
        key = n
    grouped[key] = grouped.get(key, 0) + v
grouped = pd.Series(grouped).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(grouped.index, grouped.values, color="#45A878")
ax.set_xlabel("Cumulative Importance Score"); ax.set_ylabel("Original Feature")
ax.set_title("Random Forest Feature Importance (Grouped by Original Feature)")
plt.tight_layout(); plt.savefig("figures/fig5_feature_importance.png", dpi=200); plt.show()
print(grouped.round(3).to_dict())

## 7. Figure 6: Distributions of wins and streaks

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(df.Team_Wins_Before, bins=np.arange(-0.5, 14.5, 1), color="#4895C8", edgecolor="white")
ax[0].set_title("Team Wins Before Match"); ax[0].set_xlabel("Wins before the match"); ax[0].set_ylabel("Number of records")
ax[1].hist(df.Team_Win_Streak, bins=np.arange(-13.5, 6.5, 1), color="#4895C8", edgecolor="white")
ax[1].set_title("Team Win Streak"); ax[1].set_xlabel("Win streak (negative = losing streak)")
plt.tight_layout(); plt.savefig("figures/fig6_distributions.png", dpi=200); plt.show()
print("Records with 3 or fewer prior wins:", f"{(df.Team_Wins_Before <= 3).mean():.1%}")
print("Records with streak between -2 and +2:", f"{df.Team_Win_Streak.between(-2, 2).mean():.1%}")

## 8. Outlier check
Long losing streaks are real results, not data errors.

In [ ]:
extreme = df[df.Team_Win_Streak <= -5]
print("Records with a streak of -5 or worse:", len(extreme))
print("Teams involved:", extreme.Team.value_counts().to_dict())
print("Wins among them:", int(extreme.Match_Outcome.sum()))
print("Highest prior-win count:", int(df.Team_Wins_Before.max()))